# 1. [calculate] labour productivity
- Save to `working_yearly` new table
**Assets**
- Total assets: book value of all assets
(i.e. intangible and tangible assets, stock, current and non-currents assets)#
- Total liabilities: sum of current liabilities (i.e. loans and short-term debt, creditors and non-current liabilities (i.e. long-term financial liabilities including borrowing from credit institutions and bonds issued).
- Leverage: ratio of total liabilities to total assets.
  
**Income**
- Operating revenue (turnover): sum of net sales, other operating revenues and stock variations.
- Wage bill: renumeration_employees
- Employment: number of employees on the company’s payroll. 
- Negative turnover values. Turnover is defined as the operating revenue in FAME. In a few cases, some companies report negative turnover values. We flag (but keep) those companies reporting negative turnover values.  
   
**Productivity** 
- GVA (Lars): wage bill + EBITDA
- GVA (bottom-up): profit_loss_pretax + interest_paid + depreciation + remuneration_employees
- Productivity: GVA / employees
- Average wage: wage bill / employees
- Use lns

In [1]:
import ibis
from utils.f_0_dirs import get_data_dirs

old_table_name = "fame_yearly_kp"
new_table_name = "working_yearly"

# Initialize connection
dirs = get_data_dirs()
con = ibis.duckdb.connect(str(dirs.db_path))

# Reference the existing deflated table
fame_yearly = con.table(old_table_name)

# Calculate the new metrics using Ibis lazy evaluation
# We use ibis.ifelse to safely handle natural logarithms of negative or zero GVA
working_yearly = fame_yearly.mutate(
    gva1 = fame_yearly.wages + fame_yearly.ebitda,
    gva2 =  fame_yearly.profit_loss_pretax +
            fame_yearly.interest_paid +
            fame_yearly.depreciation +
            fame_yearly.remuneration_employees
).mutate(
    gva1_per_worker = ibis._.gva1 / fame_yearly.employees,
    gva2_per_worker = ibis._.gva2 / fame_yearly.employees,
    average_wage = fame_yearly.wages / fame_yearly.employees
)

# Verify the final materialized table
table_t_working = ibis.memtable(working_yearly)
print(f"\nSample of {new_table_name}:")
display(table_t_working.sample(0.0001).execute())

IOException: IO Error: Cannot open file "C:\Users\lazyst\Files\ucl\Dissertation\build\output\fame_data.duckdb": The process cannot access the file because it is being used by another process.

File is already open in 
C:\Users\lazyst\AppData\Local\python\pythoncore-3.14-64\python.exe (PID 21188)

In [ ]:
working_yearly_skinny = table_t_working.select(
    "registered_number", "year",
    "employees", "fixed_total", "total_assets",
    "average_wage", "gva1", "gva2",
    "gva1_per_worker", "gva2_per_worker"
)

print(f"✅ Inserting columns into new '{new_table_name}' table: {working_yearly_skinny.columns}")
con.create_table(new_table_name, working_yearly_skinny, overwrite=True)

# Verify the final materialized table
final_table = con.table(new_table_name)
row_count = final_table.count().execute()
col_count = len(final_table.columns)

print(f"✅ Materialized '{new_table_name}' table.")
print(f"📊 Number of rows: {row_count:,}")
print(f"📊 Number of columns: {col_count}")
print(f"\nHead of {new_table_name}:")
display(final_table.sample(200 / row_count).execute())

✅ Inserting columns into new 'working_yearly' table: ('registered_number', 'year', 'employees', 'fixed_total', 'total_assets', 'average_wage', 'gva1', 'gva2', 'gva1_per_worker', 'gva2_per_worker')
✅ Materialized 'working_yearly' table.
📊 Number of rows: 1,102,222
📊 Number of columns: 10

Head of working_yearly:


,registered_number,year,employees,fixed_total,total_assets,average_wage,gva1,gva2,gva1_per_worker,gva2_per_worker
0,01341453,2006,272,6394.816388,49315.449165,45.919072,18943.139460,NaN,69.643895,NaN
1,01625819,2006,13,234.588542,330.403577,10.360599,-117.986296,NaN,-9.075869,NaN
2,04337458,2006,120,239.840300,12118.257464,40.493669,12146.310190,NaN,101.219252,NaN
3,03416057,2006,54,2742.759768,4577.345244,28.221053,603.789642,NaN,11.181290,NaN
4,00117548,2006,31,974.358118,11115.250379,23.752833,5258.116290,5206.190505,169.616655,167.941629
...,...,...,...,...,...,...,...,...,...,...
187,13057610,2024,142,526.432000,33192.344000,49.183915,9049.889000,NaN,63.731613,NaN
188,14761237,2024,12,NaN,331.873000,10.333417,491.389000,NaN,40.949083,NaN
189,04265102,2024,120,5617.347000,13519.155000,57.349317,8891.201000,NaN,74.093342,NaN
190,00290012,2024,16,0.144000,7779.859000,59.743687,1408.274000,1437.145000,88.017125,89.821562


# [calculate] 2. TFP

$$\ln(Y_{it}) = \alpha_i + \gamma_t + \beta_K \ln(K_{it}) + \beta_L \ln(L_{it}) + \varepsilon_{it}$$
- $Y_{it}$: `gva1` or `gva2`
- $K_{it}$: `fixed_total` or `total_assets`
- $L_{it}$: `employees`  
### Capital choice
- `fixed_total` = `tangibles` + `intangibles` + `investments_other`, representing different types of capitals
- `total_assets` = `fixed_total` + `current_assets`, which includes non-productive current_assets (e.g. cash, stock, debtors) and productive current_assets (e.g. stock of raw materials, work in progress and finished goods).

In [ ]:
import ibis
from ibis import _
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS
from utils.f_0_dirs import get_data_dirs

dirs = get_data_dirs(segment="descriptives")
con = ibis.duckdb.connect(str(dirs.db_path))
table_working = con.table("working_yearly")
table_results = table_working.select("registered_number", "year")

# 3. Define the 4 model setups to iterate through
# Varying Y (gva1 vs gva2) and K (fixed_total vs tangibles)
models = {
    'tfp1': {'Y': 'gva1', 'K': 'total_assets', 'L': 'employees'},
    'tfp2': {'Y': 'gva2', 'K': 'total_assets', 'L': 'employees'},
    'tfp3': {'Y': 'gva1', 'K': 'fixed_total', 'L': 'employees'}
}

# Dictionary to store the parameter outputs (\beta_K, \beta_L) and model summaries
parameter_tables = {}

for name, mod in models.items():

    # Mutate to dynamically log-transform all columns, adding ln_ prefix to column name
    table_skinny = (
        table_working
        .select(["registered_number", "year"] + list(mod.values()))
        .rename(mod)
    )
    table_start = table_skinny
    for col in ['Y', 'K', 'L']:
        table_filtered = table_start.filter(_[col] > 0)
        table_logged = table_filtered.mutate(**{f'ln_{col}': np.log(_[col]) }) # type: ignore
        table_start = table_logged

    # 1. Execute into a Pandas DataFrame and set the MultiIndex for linearmodels
    df_model = table_start.execute().set_index(['registered_number', 'year'])

    # Define Endogenous (Y) and Exogenous (X) variables
    Y = df_model['ln_Y']
    X = sm.add_constant(df_model[['ln_K', 'ln_L']])
    
    # 2. Estimate the model with Firm and Year Fixed Effects
    mod_ols = PanelOLS(Y, X, entity_effects=True, time_effects=True)
    
    # Fit model with firm-clustered standard errors
    res = mod_ols.fit(cov_type='clustered', cluster_entity=True)
    
    # Extract \beta_K and \beta_L
    beta_K = res.params['ln_K']
    beta_L = res.params['ln_L']
    
    # 3. Calculate firm-year specific TFP (Solow Residual) inside the Ibis pipeline
    # TFP_it = ln(Y_it) - \beta_K*ln(K_it) - \beta_L*ln(L_it)
    table_with_tfp = table_start.mutate(
        **{name: _['ln_Y'] - (beta_K * _['ln_K']) - (beta_L * _['ln_L'])}
    )
    
    # 4. Join the calculated TFP column back to the main dataframe
    # We select only the keys and the new TFP column to avoid duplicating ln_ columns
    table_results = (
        table_results
        .left_join(
            table_with_tfp,
            ["registered_number", "year"],
            rname='{name}_' + name
        )
        .drop("registered_number_" + name, "year_" + name) # Drop duplicate join keys
    )
    
    # 5. Store the results and parameters
    parameter_tables[name] = {
        'beta_K': beta_K,
        'beta_L': beta_L,
        # Safely extract time effects if they exist
        'gamma_t': res.estimated_effects.xs('time_effects', level=1) if 'time_effects' in res.estimated_effects.index.names else None, 
        'summary': res.summary
    }
    print(f"✅ Model '{name}' estimated: beta_K={beta_K:.4f}, beta_L={beta_L:.4f}")

print(f"Panel regressions complete. {len(models)} TFP variants added to the dataframe.")

✅ Model 'tfp1' estimated: beta_K=0.2852, beta_L=0.6174
✅ Model 'tfp2' estimated: beta_K=0.2642, beta_L=0.6114
✅ Model 'tfp3' estimated: beta_K=0.0652, beta_L=0.7194
Panel regressions complete. 3 TFP variants added to the dataframe.


In [ ]:
# From the above cell, display the revised table with the new TFP columns and the parameter estimates for each model
table_sample = table_results.sample(0.0001).execute()
display(table_sample)

# Send parameter_tables to a markdown file in dirs.output_dir to easily compare
output_file = dirs.output_dir / f"tfp_2factor_results_{name}.md"
with open(output_file, 'w') as f:
    for name, params in parameter_tables.items():
        # Write all to 1 big markdown file
        # Just write the default display(params) output to the file
        f.write(f"# Parameters for model '{name}'\n\n")
        f.write(f"## Estimated Coefficients\n")
        f.write(f"- beta_K: {params['beta_K']:.6f}\n")
        f.write(f"- beta_L: {params['beta_L']:.6f}\n")
        if params['gamma_t'] is not None:
            f.write(f"\n## Time Effects (gamma_t)\n")
            f.write(params['gamma_t'].to_markdown())
        f.write("\n\n## Model Summary\n")
        f.write(params['summary'].as_text())
        print(f"✅ Parameters for model '{name}' written to {output_file}")

# Verify that that \ln Y = \alpha_i + \gamma_t + \beta_K \ln K + \beta_L \ln L + TFP_it holds for this sample
name, mod = list(models.items())[0]
params = parameter_tables[name]
for row in table_sample.itertuples():
    # Get TFP, L, K
    tfp = row._asdict()[name]
    ln_K = row.ln_K
    ln_L = row.ln_L
    ln_Y = row.ln_Y
    if any(np.isnan([tfp, ln_K, ln_L, ln_Y])):
        print(f"Skipping row {row.Index} due to NaN values.")
        continue
    ln_Y_calc = tfp + params['beta_K'] * ln_K + params['beta_L'] * ln_L
    diff = ln_Y - ln_Y_calc

    assert np.isclose(diff, 0, atol=1e-6), (
        f"TFP calculation check failed for row {row.Index}!  \
        Expected ln_Y: {ln_Y:.6f}, Calculated ln_Y: {ln_Y_calc:.6f}, Difference: {diff:.6e}"
    )

# Count number of tfp1 and tfp2 observations in the results table (as a percentage of total rows)
# Output as a pd dataframe
total_rows = table_results.count().execute()
tfp1_count = table_results.filter(_['tfp1'].isnull() == False).count().execute()
tfp2_count = table_results.filter(_['tfp2'].isnull() == False).count().execute()
tfp3_count = table_results.filter(_['tfp3'].isnull() == False).count().execute()
tfp_counts = pd.DataFrame({
    'TFP Variant': ['tfp1', 'tfp2', 'tfp3'],
    'Count': [tfp1_count, tfp2_count, tfp3_count]
})
tfp_counts['Percentage'] = tfp_counts['Count'] / total_rows * 100
print(f"\nTFP Counts and Percentages (out of {total_rows:,} total rows):")
display(tfp_counts)

,registered_number,year,Y,K,L,ln_Y,ln_K,ln_L,tfp1,Y_tfp2,...,ln_K_tfp2,ln_L_tfp2,tfp2,Y_tfp3,K_tfp3,L_tfp3,ln_Y_tfp3,ln_K_tfp3,ln_L_tfp3,tfp3
0,02600346,2006,23170.120613,171461.802731,310.0,10.050619,12.052116,5.736572,3.060242,35457.776976,...,12.052116,5.736572,3.772386,23170.120613,5902.907436,310.0,10.050619,8.683200,5.736572,5.356775
1,00693949,2007,NaN,NaN,NaN,NaN,NaN,NaN,NaN,52338.836979,...,11.273634,6.793466,3.722762,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00345279,2007,21675.880516,91640.417533,279.0,9.983955,11.425628,5.631212,3.237990,22473.109431,...,11.425628,5.631212,3.547028,21675.880516,20119.239227,279.0,9.983955,9.909432,5.631212,5.286129
3,02012269,2008,3483.693838,10318.219302,42.0,8.155848,9.241666,3.737670,3.203446,3786.060217,...,9.241666,3.737670,3.502540,3483.693838,64.352222,42.0,8.155848,4.164371,3.737670,5.194756
4,02110715,2008,2813.495173,5133.585383,65.0,7.942183,8.543560,4.174387,2.920299,2412.000880,...,8.543560,4.174387,2.970186,2813.495173,1689.257029,65.0,7.942183,7.432044,4.174387,4.454096
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68,04051544,2020,49634.197537,72094.014466,612.0,10.812435,11.185726,6.416732,3.650593,NaN,...,NaN,NaN,NaN,49634.197537,19291.550226,612.0,10.812435,9.867422,6.416732,5.551994
69,11220670,2020,42.598679,54.931836,10.0,3.751823,4.006093,2.302585,1.184097,NaN,...,NaN,NaN,NaN,42.598679,4.831820,10.0,3.751823,1.575223,2.302585,1.992098
70,05412328,2021,5184.059201,6386.699145,158.0,8.553344,8.761973,5.062595,2.920982,NaN,...,NaN,NaN,NaN,5184.059201,4600.007894,158.0,8.553344,8.433813,5.062595,4.360801
71,10333734,2019,21590.948264,79959.749006,147.0,9.980029,11.289279,4.990433,3.668404,NaN,...,NaN,NaN,NaN,21590.948264,64479.220887,147.0,9.980029,11.074098,4.990433,5.667574


✅ Parameters for model 'tfp1' written to /mnt/c/Users/lazym/Documents/Code/dissertation/descriptives/output/tfp_2factor_results_tfp3.md
✅ Parameters for model 'tfp2' written to /mnt/c/Users/lazym/Documents/Code/dissertation/descriptives/output/tfp_2factor_results_tfp3.md
✅ Parameters for model 'tfp3' written to /mnt/c/Users/lazym/Documents/Code/dissertation/descriptives/output/tfp_2factor_results_tfp3.md
Skipping row 1 due to NaN values.
Skipping row 32 due to NaN values.
Skipping row 47 due to NaN values.
Skipping row 57 due to NaN values.
Skipping row 61 due to NaN values.
Skipping row 72 due to NaN values.

TFP Counts and Percentages (out of 1,102,222 total rows):


,TFP Variant,Count,Percentage
0,tfp1,1020691,92.603033
1,tfp2,592751,53.777823
2,tfp3,983564,89.234655


In [ ]:
preferred_model = 'tfp1'
# Add the tfp1 column from this table_results to the working_yearly table in the database
# Rename as 'tfp'
table_with_tfp = (
    table_working
    .left_join(
        table_results.select(['registered_number', 'year', preferred_model]),
        ['registered_number', 'year']
    )
    # Rename the preferred TFP column to 'tfp' for clarity
    .rename(tfp=preferred_model)
    .drop("registered_number_right", "year_right")
)
display(table_with_tfp.sample(0.0001).execute())

,registered_number,year,employees,fixed_total,total_assets,average_wage,gva1,gva2,gva1_per_worker,gva2_per_worker,tfp
0,SC238899,2006,18,23.280537,923.455002,60.953045,1369.167845,NaN,76.064880,NaN,3.483447
1,02450574,2006,536,331.483569,87656.477385,54.586498,35771.737091,NaN,66.738315,NaN,3.348895
2,01522328,2006,18,42.569044,32070.256449,42.086814,-26742.039774,-22649.726932,-1485.668876,-1258.318163,NaN
3,04458603,2006,174,2796.067247,58901.759804,38.510354,19512.206151,22552.925994,112.139116,129.614517,3.550682
4,03830499,2007,80,1936.822594,6833.919835,37.644543,3653.123573,3843.839087,45.664045,48.047989,2.971430
...,...,...,...,...,...,...,...,...,...,...,...
97,01280133,2023,179,164955.796000,166105.969000,43.317754,12703.673916,9864.946818,70.970245,55.111435,2.807146
98,03965910,2023,98,0.386508,1444.532090,11.745887,1564.908799,NaN,15.968457,NaN,2.443556
99,02804879,2023,58,0.158967,9499.554142,106.041507,7599.034168,NaN,131.017830,NaN,3.807932
100,15052782,2024,582,112288.054000,135531.273000,20.028216,6447.393000,NaN,11.077995,NaN,1.459820


In [ ]:
overwrite = True
if overwrite:
    # Safe overwrite of the working_yearly table with the new tfp column
    con.create_table("working_yearly_temp", table_with_tfp)
    con.create_table("working_yearly", con.table("working_yearly_temp"), overwrite=True)
    con.drop_table("working_yearly_temp")

    # Verify schema and rows of the final working_yearly table
    final_table = con.table("working_yearly")
    row_count = final_table.count().execute()
    col_count = len(final_table.columns)
    tfp_col_exists = 'tfp' in final_table.columns
    tfp_nan_count = final_table.filter(_['tfp'].isnull() == True).count().execute() if tfp_col_exists else None
    print(f"\nFinal 'working_yearly' table verification:"
        f"\n- Rows: {row_count:,}"
        f"\n- Columns: {col_count:,}"
        f"\n- TFP column exists: {tfp_col_exists}"
        f"\n- TFP NaN count: {tfp_nan_count:,}")


Final 'working_yearly' table verification:
- Rows: 1128490
- Columns: 11
- TFP column exists: True
- TFP NaN count: 83326


In [ ]:
con.raw_sql("CHECKPOINT;")
con.disconnect()